In [16]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.15f' % x)  # Exibe 5 casas decimais

In [17]:
gdf = gpd.read_file('../../dataset/shp/clusters.shp')

In [18]:
fcu = gpd.read_file('/Volumes/ssd/Doutorado/DATASET/IBGE/Favelas e Comunidades Urbanas - 2022/poligonos/fcu_blank/qg_2022_670_fcu_agreg.shp', )
fcu.columns = ['CD_FCU', 'NM_FCU', 'CD_UF', 'NM_UF', 'SIGLA_UF', 'CD_MUN', 'NM_MUN', 'geometry']
fcu['CD_FCU'] = fcu['CD_FCU'].astype(int)
fcu['CD_MUN'] = fcu['CD_MUN'].astype(int)


In [19]:
mun = gpd.read_file('/Volumes/ssd/Doutorado/DATASET/IBGE/BR_Localidades_2010_v1/BR_Localidades_2010_v1.shp')
mun = mun[mun['NM_CATEGOR'] == 'CIDADE']
mun['CD_GEOCODM'] = mun['CD_GEOCODM'].astype(int)

In [20]:
# for in fcu, get the centroid, the mun centroid and the distance between them
fcu = fcu.to_crs(epsg=3857)
mun = mun.to_crs(epsg=3857)
fcu['distance'] = 0

for index, row in fcu.iterrows():
    centroid = row.geometry.centroid
    # print(centroid)
    mun_geom = mun[mun['CD_GEOCODM'] == row['CD_MUN']]
    mun_centroid = mun_geom.iloc[0].geometry.centroid
    # print(mun_centroid)
    distance = centroid.distance(mun_centroid)
    
    # set distance to row
    fcu.at[index, 'distance'] = distance

fcu['dist_km'] = fcu['distance'] / 1000
fcu["dist_norm"] = fcu.groupby("CD_MUN")["dist_km"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)

/var/folders/5l/x3ysgz0n0mvd7msxlmn75zhw0000gp/T/ipykernel_79767/4183837124.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '23646.846826038814' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fcu.at[index, 'distance'] = distance


In [21]:
fcu[fcu['CD_MUN'] == 1100205]

,CD_FCU,NM_FCU,CD_UF,NM_UF,SIGLA_UF,CD_MUN,NM_MUN,geometry,distance,dist_km,dist_norm
19,11002050023,Teixeirão,11,Rondônia,RO,1100205,Porto Velho,"POLYGON ((-7105862.355 -976691.522, -7106092.8...",3368.922677269234555,3.368922677269234,0.029679010004813
608,11002050035,Costa e Silva,11,Rondônia,RO,1100205,Porto Velho,"POLYGON ((-7112756.159 -976369.308, -7112754.7...",8075.558640322721658,8.075558640322722,0.085715731351987
885,11002050002,Nova Esperança,11,Rondônia,RO,1100205,Porto Velho,"POLYGON ((-7109842.438 -975113.295, -7109818.5...",6416.604482886934420,6.416604482886934,0.065964394605804
1004,11002050010,Jardim Santana IV,11,Rondônia,RO,1100205,Porto Velho,"POLYGON ((-7103873.821 -979422.246, -7103865.4...",1429.616088787476201,1.429616088787476,0.006589826188907
1025,11002050022,São Sebastião III,11,Rondônia,RO,1100205,Porto Velho,"POLYGON ((-7114834.694 -976685.496, -7114792.0...",10162.643809704739397,10.162643809704740,0.110564351679234
...,...,...,...,...,...,...,...,...,...,...,...
10425,11002050055,Roque C,11,Rondônia,RO,1100205,Porto Velho,"POLYGON ((-7112076.487 -980871.563, -7112260.7...",6678.721253989412617,6.678721253989413,0.069085129792887
10728,11002050043,Cohab II,11,Rondônia,RO,1100205,Porto Velho,"POLYGON ((-7111043.442 -983394.267, -7111213.1...",5986.389942468080335,5.986389942468080,0.060842304750077
10946,11002050064,Jardim Santana I,11,Rondônia,RO,1100205,Porto Velho,"POLYGON ((-7103826.288 -977725.362, -7103817.3...",2828.325204839405615,2.828325204839405,0.023242712368595
11372,11002050065,Socialista III,11,Rondônia,RO,1100205,Porto Velho,"POLYGON ((-7104874.338 -979129.904, -7104879.9...",1115.285822806147053,1.115285822806147,0.002847442528041


In [22]:
cities = gpd.read_file('/Volumes/ssd/Doutorado/DATASET/IBGE/BR_Municipios_2022/BR_Municipios_2022.shp')
cities = cities[['CD_MUN', 'AREA_KM2']]
cities['CD_MUN'] = cities['CD_MUN'].astype(int)
cities

,CD_MUN,AREA_KM2
0,1100015,7067.127000000000407
1,1100023,4426.570999999999913
2,1100031,1314.352000000000089
3,1100049,3793.000000000000000
4,1100056,2783.300000000000182
...,...,...
5567,5222005,954.115000000000009
5568,5222054,733.793999999999983
5569,5222203,1052.593000000000075
5570,5222302,2181.592999999999847


In [23]:
join_data = fcu.merge(cities, on='CD_MUN', how='left')
join_data = join_data.merge(gdf.drop(columns=['geometry']), on='CD_FCU', how='left')
join_data

,CD_FCU,NM_FCU,CD_UF,NM_UF,SIGLA_UF,CD_MUN,NM_MUN,geometry,distance,dist_km,dist_norm,AREA_KM2,hh_density,age_index,median_age,sex_ratio,literate,race_wht,race_blk,race_ylw,race_brn,race_ind,race_unk,depriv_idx,priv_hh,coll_hh,hh_type_1,hh_type_2,hh_type_3,hh_type_4,hh_type_5,hh_type_6,sanit_1,sanit_2,sanit_3,sanit_4,sanit_5,sanit_6,sanit_7,no_bath,excl_bath,shrd_bath,toilet,no_toilet,wtr_pipe,wtr_semi,no_wtr_pip,wtr_dist,wtr_deep,wtr_shal,wtr_spring,wtr_truck,wtr_rain,wtr_river,wtr_oth,wtr_area,veg_area,risk_area,slope_10,slope_20,slope_30,slope_30p,hand_6p,hand_3_6,hand_0_3,SO2,CO,O3,NO2,waste_coll,demo_dens,est_dens,est_edu,est_health,est_relig,est_agri,est_other,est_const,intra_veg,prim_hosp,inp_hosp,wtr_idx,sanit_idx,living_are,durable_ho,cluster_3
0,35503080069,Jardim Miliunas,35,São Paulo,SP,3550308,São Paulo,"POLYGON ((-5162009.388 -2692597.734, -5162042....",23646.846826038814470,23.646846826038814,0.533158552343469,1521.201999999999998,2.956632653061225,44.020000000000003,29.000000000000000,99.000000000000000,92.890000000000001,32.609999999999999,17.340000000000000,0.000000000000000,50.039999999999999,0.000000000000000,0.000000000000000,0.593881394151810,100.000000000000000,0.000000000000000,0.318900000000000,0.002600000000000,0.000000000000000,0.678600000000000,0.000000000000000,0.000000000000000,0.206600000000000,0.000000000000000,0.030600000000000,0.000000000000000,0.000000000000000,0.762800000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.994900000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.005100000000000,0.000000000000000,0.000000000000000,0.000000000000000,50.965558831959356,49.034441168040658,0.000000000000000,0.000000000000000,35.578202056669660,26.962626321230797,37.459171622099561,0.000226338812134,0.032435898459901,0.117353676384719,0.000138925792410,98.719999999999999,45062.209999999999127,884.615384615384642,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,100.000000000000000,100.000000000000000,0.000000000000000,752.192509434099634,2071.724078601533165,0.997450127493625,0.451028037383178,0.681207545029720,0.321535692861428,1.000000000000000
1,42024040016,Rua Araranguá,42,Santa Catarina,SC,4202404,Blumenau,"POLYGON ((-5460228.537 -3115389.701, -5460231....",2544.012189292799121,2.544012189292799,0.080353569412576,518.619000000000028,2.809917355371901,60.270000000000003,34.000000000000000,102.000000000000000,96.629999999999995,79.709999999999994,4.120000000000000,0.000000000000000,16.180000000000000,0.000000000000000,0.000000000000000,-1.339049333599375,100.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.958700000000000,0.000000000000000,0.033100000000000,0.000000000000000,0.008300000000000,0.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,3.783948041683024,100.000000000000000,0.120335888401926,9.787379757137918,12.635362314477181,77.456922039982985,88.808712208672446,6.551667421570913,4.639620369756640,0.000128408715662,0.030195952437025,0.119900281982149,0.000063380103705,100.000000000000000,2916.000000000000000,8.547008547008547,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,100.000000000000000,0.000000000000000,886.945621266084572,281.660799669791857,1167.193154171149445,1.000000000000000,0.978808411214953,0.872972435449063,0.999800039992001,0.000000000000000
2,33045570554,Tiqui,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,"POLYGON ((-4840464.531 -2618564.009, -4840451....",28454.

In [24]:
tabela9887_2 = pd.read_excel('/Volumes/ssd/Doutorado/DATASET/IBGE/Favelas e Comunidades Urbanas - 2022/tabelas/População/9887/tabela9887_pop.xlsx', skiprows=4, skipfooter=1)
tabela9887_2.columns = ['CD_FCU', 'NM_FCU', 'pessoas']
join_data = join_data.merge(tabela9887_2, on='CD_FCU', how='left')
join_data

,CD_FCU,NM_FCU_x,CD_UF,NM_UF,SIGLA_UF,CD_MUN,NM_MUN,geometry,distance,dist_km,dist_norm,AREA_KM2,hh_density,age_index,median_age,sex_ratio,literate,race_wht,race_blk,race_ylw,race_brn,race_ind,race_unk,depriv_idx,priv_hh,coll_hh,hh_type_1,hh_type_2,hh_type_3,hh_type_4,hh_type_5,hh_type_6,sanit_1,sanit_2,sanit_3,sanit_4,sanit_5,sanit_6,sanit_7,no_bath,excl_bath,shrd_bath,toilet,no_toilet,wtr_pipe,wtr_semi,no_wtr_pip,wtr_dist,wtr_deep,wtr_shal,wtr_spring,wtr_truck,wtr_rain,wtr_river,wtr_oth,wtr_area,veg_area,risk_area,slope_10,slope_20,slope_30,slope_30p,hand_6p,hand_3_6,hand_0_3,SO2,CO,O3,NO2,waste_coll,demo_dens,est_dens,est_edu,est_health,est_relig,est_agri,est_other,est_const,intra_veg,prim_hosp,inp_hosp,wtr_idx,sanit_idx,living_are,durable_ho,cluster_3,NM_FCU_y,pessoas
0,35503080069,Jardim Miliunas,35,São Paulo,SP,3550308,São Paulo,"POLYGON ((-5162009.388 -2692597.734, -5162042....",23646.846826038814470,23.646846826038814,0.533158552343469,1521.201999999999998,2.956632653061225,44.020000000000003,29.000000000000000,99.000000000000000,92.890000000000001,32.609999999999999,17.340000000000000,0.000000000000000,50.039999999999999,0.000000000000000,0.000000000000000,0.593881394151810,100.000000000000000,0.000000000000000,0.318900000000000,0.002600000000000,0.000000000000000,0.678600000000000,0.000000000000000,0.000000000000000,0.206600000000000,0.000000000000000,0.030600000000000,0.000000000000000,0.000000000000000,0.762800000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.994900000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.005100000000000,0.000000000000000,0.000000000000000,0.000000000000000,50.965558831959356,49.034441168040658,0.000000000000000,0.000000000000000,35.578202056669660,26.962626321230797,37.459171622099561,0.000226338812134,0.032435898459901,0.117353676384719,0.000138925792410,98.719999999999999,45062.209999999999127,884.615384615384642,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,100.000000000000000,100.000000000000000,0.000000000000000,752.192509434099634,2071.724078601533165,0.997450127493625,0.451028037383178,0.681207545029720,0.321535692861428,1.000000000000000,Jardim Miliunas - São Paulo (SP),1159
1,42024040016,Rua Araranguá,42,Santa Catarina,SC,4202404,Blumenau,"POLYGON ((-5460228.537 -3115389.701, -5460231....",2544.012189292799121,2.544012189292799,0.080353569412576,518.619000000000028,2.809917355371901,60.270000000000003,34.000000000000000,102.000000000000000,96.629999999999995,79.709999999999994,4.120000000000000,0.000000000000000,16.180000000000000,0.000000000000000,0.000000000000000,-1.339049333599375,100.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.958700000000000,0.000000000000000,0.033100000000000,0.000000000000000,0.008300000000000,0.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,3.783948041683024,100.000000000000000,0.120335888401926,9.787379757137918,12.635362314477181,77.456922039982985,88.808712208672446,6.551667421570913,4.639620369756640,0.000128408715662,0.030195952437025,0.119900281982149,0.000063380103705,100.000000000000000,2916.000000000000000,8.547008547008547,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,100.000000000000000,0.000000000000000,886.945621266084572,281.660799669791857,1167.193154171149445,1.000000000000000,0.978808411214953,0.872972435449063,0.999800039992001,0.000000000000000,Rua Araranguá - Blumenau (SC),340
2,33045570554,Tiqui,33,Rio de J

In [25]:
join_data = join_data.drop(columns=['NM_FCU_y'])
join_data.columns =      ['CD_FCU', 'NM_FCU',   'CD_UF', 'NM_UF', 'SIGLA_UF', 'CD_MUN', 'NM_MUN', 'geometry', 'distance', 'dist_km', 'dist_norm', 'area',    'hh_density', 'age_idx', 'median_age', 'sex_ratio', 'literate', 'race_wht', 'race_blk', 'race_ylw', 'race_brn', 'race_ind', 'race_unk', 'depriv_idx', 'priv_hh', 'coll_hh', 'hh_type_1', 'hh_type_2', 'hh_type_3', 'hh_type_4', 'hh_type_5', 'hh_type_6', 'sanit_1', 'sanit_2', 'sanit_3', 'sanit_4', 'sanit_5', 'sanit_6', 'sanit_7', 'no_bath', 'excl_bath', 'shrd_bath', 'toilet', 'no_toilet', 'wtr_pipe', 'wtr_semi', 'no_wtr_pip', 'wtr_dist', 'wtr_deep', 'wtr_shal', 'wtr_spring', 'wtr_truck', 'wtr_rain', 'wtr_river', 'wtr_oth', 'wtr_area', 'veg_area', 'risk', 'slope_10', 'slope_20', 'slope_30', 'slope_30p', 'hand_6p', 'hand_3_6', 'hand_0_3', 'SO2', 'CO', 'O3', 'NO2', 'waste_coll', 'demo_dens', 'est_dens', 'est_edu', 'est_health', 'est_relig', 'est_agri', 'est_other', 'est_const', 'intra_veg', 'prim_hosp', 'inp_hosp', 'wtr_idx', 'sanit_idx', 'liv_a_idx', 'hh_idx', 'cluster', 'population']
join_data

,CD_FCU,NM_FCU,CD_UF,NM_UF,SIGLA_UF,CD_MUN,NM_MUN,geometry,distance,dist_km,dist_norm,area,hh_density,age_idx,median_age,sex_ratio,literate,race_wht,race_blk,race_ylw,race_brn,race_ind,race_unk,depriv_idx,priv_hh,coll_hh,hh_type_1,hh_type_2,hh_type_3,hh_type_4,hh_type_5,hh_type_6,sanit_1,sanit_2,sanit_3,sanit_4,sanit_5,sanit_6,sanit_7,no_bath,excl_bath,shrd_bath,toilet,no_toilet,wtr_pipe,wtr_semi,no_wtr_pip,wtr_dist,wtr_deep,wtr_shal,wtr_spring,wtr_truck,wtr_rain,wtr_river,wtr_oth,wtr_area,veg_area,risk,slope_10,slope_20,slope_30,slope_30p,hand_6p,hand_3_6,hand_0_3,SO2,CO,O3,NO2,waste_coll,demo_dens,est_dens,est_edu,est_health,est_relig,est_agri,est_other,est_const,intra_veg,prim_hosp,inp_hosp,wtr_idx,sanit_idx,liv_a_idx,hh_idx,cluster,population
0,35503080069,Jardim Miliunas,35,São Paulo,SP,3550308,São Paulo,"POLYGON ((-5162009.388 -2692597.734, -5162042....",23646.846826038814470,23.646846826038814,0.533158552343469,1521.201999999999998,2.956632653061225,44.020000000000003,29.000000000000000,99.000000000000000,92.890000000000001,32.609999999999999,17.340000000000000,0.000000000000000,50.039999999999999,0.000000000000000,0.000000000000000,0.593881394151810,100.000000000000000,0.000000000000000,0.318900000000000,0.002600000000000,0.000000000000000,0.678600000000000,0.000000000000000,0.000000000000000,0.206600000000000,0.000000000000000,0.030600000000000,0.000000000000000,0.000000000000000,0.762800000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.994900000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.005100000000000,0.000000000000000,0.000000000000000,0.000000000000000,50.965558831959356,49.034441168040658,0.000000000000000,0.000000000000000,35.578202056669660,26.962626321230797,37.459171622099561,0.000226338812134,0.032435898459901,0.117353676384719,0.000138925792410,98.719999999999999,45062.209999999999127,884.615384615384642,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,100.000000000000000,100.000000000000000,0.000000000000000,752.192509434099634,2071.724078601533165,0.997450127493625,0.451028037383178,0.681207545029720,0.321535692861428,1.000000000000000,1159
1,42024040016,Rua Araranguá,42,Santa Catarina,SC,4202404,Blumenau,"POLYGON ((-5460228.537 -3115389.701, -5460231....",2544.012189292799121,2.544012189292799,0.080353569412576,518.619000000000028,2.809917355371901,60.270000000000003,34.000000000000000,102.000000000000000,96.629999999999995,79.709999999999994,4.120000000000000,0.000000000000000,16.180000000000000,0.000000000000000,0.000000000000000,-1.339049333599375,100.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.958700000000000,0.000000000000000,0.033100000000000,0.000000000000000,0.008300000000000,0.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,3.783948041683024,100.000000000000000,0.120335888401926,9.787379757137918,12.635362314477181,77.456922039982985,88.808712208672446,6.551667421570913,4.639620369756640,0.000128408715662,0.030195952437025,0.119900281982149,0.000063380103705,100.000000000000000,2916.000000000000000,8.547008547008547,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,100.000000000000000,0.000000000000000,886.945621266084572,281.660799669791857,1167.193154171149445,1.000000000000000,0.978808411214953,0.872972435449063,0.999800039992001,0.000000000000000,340
2,33045570554,Tiqui,33,Rio de Janeiro,RJ,3304557,Rio de Janeiro,"POLYGON ((-4840464.531 -2618564.009, -4840451....",2845

In [26]:
area_density = pd.read_excel('../../dataset/area_density.xlsx', skiprows=3, skipfooter=1)
area_density.columns = ['CD_FCU', 'name', 'area_km2', 'km', 'density_km2', 'dens_km']
area_density.drop(columns=['name', 'km', 'dens_km'], inplace=True)
area_density['CD_FCU'] = area_density['CD_FCU'].astype(int)
area_density

,CD_FCU,area_km2,density_km2
0,11000490002,0.037000000000000,1388.809999999999945
1,11000490006,0.145000000000000,3400.159999999999854
2,11001060003,0.269000000000000,2338.940000000000055
3,11001060004,0.294000000000000,1835.130000000000109
4,11001060005,0.605000000000000,1751.039999999999964
...,...,...,...
12343,53001080089,10.493000000000000,1123.279999999999973
12344,53001080090,0.216000000000000,1483.119999999999891
12345,53001080094,0.165000000000000,2242.559999999999945
12346,53001080095,0.593000000000000,1677.039999999999964


In [27]:
final_version = join_data.merge(area_density, on='CD_FCU', how='left')
final_version

,CD_FCU,NM_FCU,CD_UF,NM_UF,SIGLA_UF,CD_MUN,NM_MUN,geometry,distance,dist_km,dist_norm,area,hh_density,age_idx,median_age,sex_ratio,literate,race_wht,race_blk,race_ylw,race_brn,race_ind,race_unk,depriv_idx,priv_hh,coll_hh,hh_type_1,hh_type_2,hh_type_3,hh_type_4,hh_type_5,hh_type_6,sanit_1,sanit_2,sanit_3,sanit_4,sanit_5,sanit_6,sanit_7,no_bath,excl_bath,shrd_bath,toilet,no_toilet,wtr_pipe,wtr_semi,no_wtr_pip,wtr_dist,wtr_deep,wtr_shal,wtr_spring,wtr_truck,wtr_rain,wtr_river,wtr_oth,wtr_area,veg_area,risk,slope_10,slope_20,slope_30,slope_30p,hand_6p,hand_3_6,hand_0_3,SO2,CO,O3,NO2,waste_coll,demo_dens,est_dens,est_edu,est_health,est_relig,est_agri,est_other,est_const,intra_veg,prim_hosp,inp_hosp,wtr_idx,sanit_idx,liv_a_idx,hh_idx,cluster,population,area_km2,density_km2
0,35503080069,Jardim Miliunas,35,São Paulo,SP,3550308,São Paulo,"POLYGON ((-5162009.388 -2692597.734, -5162042....",23646.846826038814470,23.646846826038814,0.533158552343469,1521.201999999999998,2.956632653061225,44.020000000000003,29.000000000000000,99.000000000000000,92.890000000000001,32.609999999999999,17.340000000000000,0.000000000000000,50.039999999999999,0.000000000000000,0.000000000000000,0.593881394151810,100.000000000000000,0.000000000000000,0.318900000000000,0.002600000000000,0.000000000000000,0.678600000000000,0.000000000000000,0.000000000000000,0.206600000000000,0.000000000000000,0.030600000000000,0.000000000000000,0.000000000000000,0.762800000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.994900000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.005100000000000,0.000000000000000,0.000000000000000,0.000000000000000,50.965558831959356,49.034441168040658,0.000000000000000,0.000000000000000,35.578202056669660,26.962626321230797,37.459171622099561,0.000226338812134,0.032435898459901,0.117353676384719,0.000138925792410,98.719999999999999,45062.209999999999127,884.615384615384642,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,100.000000000000000,100.000000000000000,0.000000000000000,752.192509434099634,2071.724078601533165,0.997450127493625,0.451028037383178,0.681207545029720,0.321535692861428,1.000000000000000,1159,0.026000000000000,45062.209999999999127
1,42024040016,Rua Araranguá,42,Santa Catarina,SC,4202404,Blumenau,"POLYGON ((-5460228.537 -3115389.701, -5460231....",2544.012189292799121,2.544012189292799,0.080353569412576,518.619000000000028,2.809917355371901,60.270000000000003,34.000000000000000,102.000000000000000,96.629999999999995,79.709999999999994,4.120000000000000,0.000000000000000,16.180000000000000,0.000000000000000,0.000000000000000,-1.339049333599375,100.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.958700000000000,0.000000000000000,0.033100000000000,0.000000000000000,0.008300000000000,0.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,1.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,3.783948041683024,100.000000000000000,0.120335888401926,9.787379757137918,12.635362314477181,77.456922039982985,88.808712208672446,6.551667421570913,4.639620369756640,0.000128408715662,0.030195952437025,0.119900281982149,0.000063380103705,100.000000000000000,2916.000000000000000,8.547008547008547,0.000000000000000,0.000000000000000,0.000000000000000,0.000000000000000,100.000000000000000,0.000000000000000,886.945621266084572,281.660799669791857,1167.193154171149445,1.000000000000000,0.978808411214953,0.872972435449063,0.999800039992001,0.000000000000000,340,0.117000000000000,2916.000000000000000
2,33045570554,Tiqui,

In [28]:
# drop rows with missing cluster
final_version = final_version[final_version['cluster'].notna()]

# drop rows with missing wtr_area
final_version_no_missing_values = final_version[final_version['wtr_area'].notna()]

In [29]:
# save shp
final_version.to_file('../../dataset/shp/final_version.shp', driver='ESRI Shapefile')

final_version_no_missing_values.to_file('../../dataset/shp/final_version_no_missing_values.shp', driver='ESRI Shapefile')

/var/folders/5l/x3ysgz0n0mvd7msxlmn75zhw0000gp/T/ipykernel_79767/149386657.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  final_version.to_file('../../dataset/shp/final_version.shp', driver='ESRI Shapefile')
/Users/joaosilva/workspace/phd/classify-slum/venv/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'density_km2' to 'density_km'
  ogr_write(
/var/folders/5l/x3ysgz0n0mvd7msxlmn75zhw0000gp/T/ipykernel_79767/149386657.py:4: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  final_version_no_missing_values.to_file('../../dataset/shp/final_version_no_missing_values.shp', driver='ESRI Shapefile')
/Users/joaosilva/workspace/phd/classify-slum/venv/lib/python3.13/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'density_km2' to 'density_km'
  ogr_write(


In [30]:
# to json
df = pd.DataFrame(final_version)
# drop geometry
df = df.drop(columns=['geometry'])
# save json
df.to_json('../../dataset/json/final_version.json', orient='records')